# Scenario Demo
This notebook shows how to **create**, **save**, **load**, and **switch** between reproducible CARLA scenarios using `WorldManager` and `Scenario`.

In [9]:
# Launch CARLA (skip if already running)
import subprocess
subprocess.Popen("CarlaUE4.exe", shell=True)

<Popen: returncode: None args: 'CarlaUE4.exe'>

## 0. List available maps
Query the running CARLA server for all available maps.

In [14]:
import carla

client = carla.Client("127.0.0.1", 2000)
client.set_timeout(30.0)

maps = client.get_available_maps()
for m in maps:
    print(m)

print([m.split("/")[-1] for m in maps])

Town01
Town01_Opt
Town02
Town02_Opt
Town03
Town03_Opt
Town04
Town04_Opt
Town05
Town05_Opt
Town06
Town06_Opt
Town07
Town07_Opt
Town10HD
Town10HD_Opt
Town11
Town12
Town13
Town15
AnnotationColorLandscape
['Town01', 'Town01_Opt', 'Town02', 'Town02_Opt', 'Town03', 'Town03_Opt', 'Town04', 'Town04_Opt', 'Town05', 'Town05_Opt', 'Town06', 'Town06_Opt', 'Town07', 'Town07_Opt', 'Town10HD', 'Town10HD_Opt', 'Town11', 'Town12', 'Town13', 'Town15', 'AnnotationColorLandscape']


## 1. Generate scenarios
Build the full scenario list using the generator and save them to disk.

In [ ]:
from MIREIA.simulation.scenarios import generate_mireia_dataset
from MIREIA.simulation.world_manager import WorldManager

scenarios = generate_mireia_dataset()
print(f"Generated {len(scenarios)} scenarios")

for scenario in scenarios:
    scenario.save()

print(len(scenarios))

scenarios_to_run = scenarios[42:55]  # Run the first 5 scenarios for demonstration
print(f"Saved scenarios to: {scenario.folder_path}")

Generated 56 scenarios
56
Saved scenarios to: d:\Projectes\TFG\MIREIA\scenarios\14D_HardRainSunset_Town04_LowVol


: 

## 2. Run pipeline for each scenario
For each scenario, load it, run a short simulation, capture sensors, record a dataset, and compose a video.

In [ ]:
from pathlib import Path
import time

include_topdown = False
include_risk_maps = False

create_videos = False


frame_stride = 5
# Frames stride of 5 and simulation delta time of 0.05s means we capture a frame every 0.25s, which is 4 frames per second.
# If we want to capture 15 minutes of data, that would be 15 * 60 * 4 = 3600 captured frames total.
num_frames = 3600 * frame_stride  # Total frames to run in the simulation (including skipped frames)

risk_map_resolution = (1024, 1024)
video_fps = 25
warmup_ticks = 50

def _format_time(seconds: float) -> str:
    minutes = seconds / 60.0
    return f"{seconds:.1f} seconds ({minutes:.1f} minutes)"

overall_start = time.time()
overall_scenarios_done = 0
frames_per_scenario = num_frames // frame_stride
total_frames_to_capture = frames_per_scenario * len(scenarios_to_run)
total_captured_frames = 0

for idx, scenario in enumerate(scenarios_to_run):
    print(f"\n=== Scenario {idx + 1}/{len(scenarios_to_run)}: {scenario.name} ===")

    scenario_start = time.time()
    wm = WorldManager(verbose=True)

    for i in range(5):  # Try up to 3 times to load the scenario
        try:
            wm.load_scenario(scenario)
            break
        except Exception as e:
            print(f"Failed to load scenario {scenario.name}: {e}")
            if i == 4:
                raise

    for _ in range(warmup_ticks):
        wm.tick()

    sensors = wm.setup_sensors()

    scenario_root = Path(wm.scenario.folder_path)
    dataset_dir = scenario_root / "dataset"
    images_dir = dataset_dir / "images"
    images_dir.mkdir(parents=True, exist_ok=True)

    wm.enable_recording(append=False, include_topdown=include_topdown, include_static_risk_image=False)

    def tick_without_log():
        wm.world.tick()
        if wm.bridge is not None:
            wm.bridge.update()

    captured_frames = 0
    for frame_id in range(num_frames):
        if frame_id % frame_stride == 0:
            rgb_path = images_dir / f"rgb_{frame_id:06d}.png"
            rel_rgb_path = str(rgb_path.relative_to(scenario_root))

            topdown_rel = ""
            if include_topdown:
                topdown_path = images_dir / f"topdown_{frame_id:06d}.png"
                topdown_rel = str(topdown_path.relative_to(scenario_root))
            risk_map_rel = ""
            if include_risk_maps:
                risk_map_path = images_dir / f"risk_{frame_id:06d}.png"
                risk_map_rel = str(risk_map_path.relative_to(scenario_root))

            def tick_and_log():
                wm.tick(
                    ground_truth_risk=None,
                    rgb_image_path=rel_rgb_path,
                    topdown_image_path=topdown_rel,
                    risk_map_image_path=risk_map_rel,
                )

            sensors.save_ego_frame(save_path=str(rgb_path), tick_fn=tick_and_log)

            if include_topdown:
                sensors.save_map_frame(save_path=str(topdown_path), tick_fn=tick_without_log)
            if include_risk_maps:
                wm.save_risk_frame_image(save_path=str(risk_map_path), resolution=risk_map_resolution)

            captured_frames += 1
            total_captured_frames += 1
            if captured_frames % 50 == 0:
                scenario_elapsed = time.time() - scenario_start
                avg_time_per_frame = scenario_elapsed / max(1, captured_frames)
                remaining_frames = frames_per_scenario - captured_frames
                est_remaining = remaining_frames * avg_time_per_frame

                overall_elapsed = time.time() - overall_start
                overall_scenarios_done = idx + 1
                overall_avg_time_per_frame = overall_elapsed / max(1, total_captured_frames)
                overall_remaining_frames = total_frames_to_capture - total_captured_frames
                overall_est_remaining = overall_remaining_frames * overall_avg_time_per_frame

                print("\n--- Progress Update ---")
                print(f"Captured frame {frame_id}/{num_frames} (scenario)")
                print(f"Captured frames {total_captured_frames}/{total_frames_to_capture} (overall)")
                print(f"Elapsed time (scenario): {_format_time(scenario_elapsed)}")
                print(f"Estimated remaining (scenario): {_format_time(est_remaining)}")
                print(f"Elapsed time (overall): {_format_time(overall_elapsed)}")
                print(f"Estimated remaining (overall): {_format_time(overall_est_remaining)}")

    print(f"Saved dataset to: {dataset_dir}")

    if create_videos:
        video_path = wm.compose_dataset_video(fps=video_fps)
        print(f"Saved video to: {video_path}")

    wm.destroy()


=== Scenario 1/15: 11A_WetCloudySunset_Town03_HighVol ===
Connecting to CARLA at 127.0.0.1:2000...
Connected. Current map: 'Carla/Maps/Town10HD_Opt'.
Loading scenario '11A_WetCloudySunset_Town03_HighVol' (map=Town03, seed=42)...
Switching map from 'Carla/Maps/Town10HD_Opt' to 'Town03'...
Weather set to 'WetCloudySunset'.
Spawned ego vehicle 'vehicle.lincoln.mkz_2020' at index None (autopilot=True)
ERROR: Spawn failed because of collision at spawn position
Spawned 79 / 80 requested vehicles.
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of collision at spawn position
ERROR: Spawn failed because of 

In [ ]:
stop
# it took 30 min for 2 out of 56 scenarios, which means around 7 hours for the whole dataset. We can speed it up by running multiple scenarios in parallel, but for now we'll just run a few for demonstration.

NameError: name 'stop' is not defined

In [ ]:
from pathlib import Path
import shutil

scenarios_root = Path("MIREIA/scenarios")
videos_out = scenarios_root / "videos"
videos_out.mkdir(parents=True, exist_ok=True)

video_name = "dataset_video.mp4"
copied = 0
for scenario_dir in scenarios_root.iterdir():
    if not scenario_dir.is_dir() or scenario_dir.name == "videos":
        continue
    src = scenario_dir / "dataset" / video_name
    if src.exists():
        dst = videos_out / f"{scenario_dir.name}_{video_name}"
        shutil.copy2(src, dst)
        copied += 1

print(f"Copied {copied} videos to: {videos_out}")

Copied 54 videos to: D:\Projectes\TFG\MIREIA\scenarios\videos


: 

: 

: 

In [ ]:
from pathlib import Path
import re
import shutil
import subprocess
import tempfile

videos_dir = Path("MIREIA/scenarios/videos")
output_path = videos_dir / "all_scenarios_grid_concat.mp4"

videos = sorted(videos_dir.glob("*_dataset_video.mp4"))
if not videos:
    raise RuntimeError(f"No videos found in {videos_dir}")

groups = {}
for video in videos:
    scenario_name = video.name.replace("_dataset_video.mp4", "")
    match = re.match(r"(\d{2})([A-D])_", scenario_name)
    if not match:
        continue
    group_key, letter = match.group(1), match.group(2)
    groups.setdefault(group_key, {})[letter] = video

temp_dir = videos_dir / "_grid_videos"
if temp_dir.exists():
    shutil.rmtree(temp_dir)
temp_dir.mkdir(parents=True, exist_ok=True)

grid_videos = []
for group_key in sorted(groups.keys()):
    slots = groups[group_key]
    if not all(k in slots for k in ("A", "B", "C", "D")):
        print(f"Skipping {group_key}: missing A/B/C/D")
        continue
    out_path = temp_dir / f"grid_{group_key}.mp4"
    cmd = [
        "ffmpeg", "-y",
        "-i", str(slots["A"]),
        "-i", str(slots["B"]),
        "-i", str(slots["C"]),
        "-i", str(slots["D"]),
        "-filter_complex",
        "[0:v][1:v][2:v][3:v]xstack=inputs=4:layout=0_0|w0_0|0_h0|w0_h0[v]",
        "-map", "[v]", "-shortest",
        "-c:v", "libx264", "-preset", "fast", "-crf", "20",
        "-an",
        str(out_path),
]
    subprocess.run(cmd, check=True)
    grid_videos.append(out_path)

if not grid_videos:
    raise RuntimeError("No complete A/B/C/D groups found for grid composition.")

with tempfile.NamedTemporaryFile(mode="w", suffix=".txt", delete=False) as f:
    for video in grid_videos:
        f.write(f"file '{video.as_posix()}'\n")
    concat_list = f.name

cmd = [
    "ffmpeg", "-y",
    "-f", "concat", "-safe", "0",
    "-i", concat_list,
    "-c:v", "libx264", "-preset", "fast", "-crf", "20",
    "-movflags", "+faststart",
    str(output_path),
]
subprocess.run(cmd, check=True)
print(f"Saved grid-concatenated video to: {output_path}")

Skipping 14: missing A/B/C/D
Saved grid-concatenated video to: D:\Projectes\TFG\MIREIA\scenarios\videos\all_scenarios_grid_concat.mp4


: 

: 

: 